# Part 1: Neural Network Fundamentals and Training Behavior Analysis

**Dataset:** Customer Churn Neural Network Dataset (`customer_churn_nn.csv`)  
**Task Type:** Binary Classification — Predict whether a customer will churn (1) or not (0)  
**Framework:** TensorFlow / Keras

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'Pandas     : {pd.__version__}')

---
## Task 1: Dataset Understanding

In [ ]:
df = pd.read_csv('customer_churn_nn.csv')

print('=' * 55)
print('  DATASET OVERVIEW')
print('=' * 55)
print(f'  Rows    : {df.shape[0]}')
print(f'  Columns : {df.shape[1]}')
print()
print('First 5 rows:')
df.head()

In [ ]:
cat_cols = ['region', 'plan_type', 'contract_type', 'payment_method']
num_cols = ['tenure_months', 'monthly_charges_inr', 'avg_login_days_per_month',
            'support_tickets_last_90_days', 'payment_delay_days', 'data_usage_gb',
            'satisfaction_score', 'last_complaint_days_ago', 'discount_percent',
            'autopay_enabled', 'referral_count']

print('--- Column Data Types ---')
print(df.dtypes)
print(f'\nCategorical features ({len(cat_cols)}): {cat_cols}')
print(f'Numerical features  ({len(num_cols)}): {num_cols}')
print("\n'customer_id' is an identifier — will be dropped before training.")
print("\nTarget: 'churn'  |  0 = Did NOT churn  |  1 = Churned")

In [ ]:
print('--- Missing Values ---')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()} — No missing data found.')

In [ ]:
print('--- Statistical Summary (Numerical Features) ---')
df[num_cols].describe().round(2)

In [ ]:
churn_counts = df['churn'].value_counts()
print('--- Target Variable: churn ---')
print(churn_counts)
print(f'\nChurn rate: {df["churn"].mean() * 100:.2f}%')
print('NOTE: Dataset is heavily imbalanced — only ~1.55% customers churned.')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Task 1 — Target Variable Distribution', fontsize=14, fontweight='bold')
labels = ['Not Churned (0)', 'Churned (1)']
colors = ['#2196F3', '#F44336']

axes[0].bar(labels, churn_counts.values, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Count of Churn vs No-Churn')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold', fontsize=12)

axes[1].pie(churn_counts.values, labels=labels, autopct='%1.2f%%',
            colors=colors, startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Churn Proportion')

plt.tight_layout()
plt.savefig('results/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Task 2: Data Preprocessing

In [ ]:
# Drop identifier
df_clean = df.drop(columns=['customer_id'])
print(f"Dropped 'customer_id'. Remaining columns: {df_clean.shape[1]}")

In [ ]:
# One-Hot Encode categorical columns
print('Unique values in categorical columns:')
for col in cat_cols:
    print(f'  {col}: {df_clean[col].unique().tolist()}')

df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=False)
# Convert boolean OHE columns to int
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f'\nShape after One-Hot Encoding: {df_encoded.shape}')

In [ ]:
# Separate features and target
X = df_encoded.drop(columns=['churn'])
y = df_encoded['churn']
print(f'Feature matrix X: {X.shape}')
print(f'Target vector  y: {y.shape}')

In [ ]:
# Train-Test Split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set : {X_train.shape[0]} samples  (churn: {y_train.sum()})')
print(f'Testing set  : {X_test.shape[0]} samples   (churn: {y_test.sum()})')

In [ ]:
# StandardScaler — fit on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('StandardScaler applied.')
print(f'Mean (feature 0, train): {X_train_scaled[:, 0].mean():.4f}')
print(f'Std  (feature 0, train): {X_train_scaled[:, 0].std():.4f}')

In [ ]:
# Class weights to handle imbalance
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: cw[0], 1: cw[1]}
print(f'Class weights: {class_weight_dict}')
print(f'The minority churn class gets ~{cw[1]/cw[0]:.0f}x more weight.')

---
## Task 3: Neural Network Model Building

In [ ]:
input_dim = X_train_scaled.shape[1]
print(f'Input dimension: {input_dim} features')

def build_model(hidden_layers=2, neurons=64, lr=0.001, activation='relu'):
    """
    Feed-forward neural network for binary churn classification.
    Input layer  -> Hidden layers (ReLU/tanh + Dropout) -> Output (Sigmoid)
    Loss: Binary Crossentropy | Optimizer: Adam
    """
    model = keras.Sequential(name='ChurnNN')
    # First hidden layer (with input shape)
    model.add(layers.Dense(neurons, activation=activation,
                           input_shape=(input_dim,),
                           kernel_regularizer=regularizers.l2(0.001)))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.3))
    # Additional hidden layers
    for _ in range(hidden_layers - 1):
        model.add(layers.Dense(max(neurons // 2, 16), activation=activation,
                               kernel_regularizer=regularizers.l2(0.001)))
        model.add(layers.Dropout(0.2))
    # Output layer — sigmoid for binary classification probability
    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 keras.metrics.AUC(name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model

model = build_model(hidden_layers=2, neurons=64, lr=0.001, activation='relu')
model.summary()

In [ ]:
print('Architecture:')
print('+' + '-'*50 + '+')
print(f'  Input layer       : {input_dim} features')
print(f'  Hidden layer 1    : 64 neurons, ReLU, BatchNorm, Dropout(0.3)')
print(f'  Hidden layer 2    : 32 neurons, ReLU, Dropout(0.2)')
print(f'  Output layer      : 1 neuron, Sigmoid')
print(f'  Loss              : Binary Crossentropy')
print(f'  Optimizer         : Adam (lr=0.001)')
print(f'  Regularizer       : L2(0.001)')
print(f'  Imbalance handling: Class weights (~63x for churn class)')
print('+' + '-'*50 + '+')

---
## Task 4: Training and Evaluation

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10,
                            restore_best_weights=True, verbose=1)

history = model.fit(
    X_train_scaled, y_train,
    epochs=80,
    batch_size=32,
    validation_split=0.15,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Task 4 — Training vs Validation Curves (Baseline)', fontsize=13, fontweight='bold')

axes[0].plot(history.history['loss'], label='Train Loss', color='#2196F3', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#F44336', linewidth=2)
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['auc'], label='Train AUC', color='#4CAF50', linewidth=2)
axes[1].plot(history.history['val_auc'], label='Val AUC', color='#FF9800', linewidth=2)
axes[1].set_title('AUC Score')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=' * 55)
print('  MODEL EVALUATION ON TEST SET')
print('=' * 55)

test_results = model.evaluate(X_test_scaled, y_test, verbose=0)
metric_names = ['Loss', 'Accuracy', 'AUC', 'Precision', 'Recall']
for name, val in zip(metric_names, test_results):
    print(f'  {name:<12}: {val:.4f}')

y_pred_prob = model.predict(X_test_scaled, verbose=0).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

print(f'\n  ROC-AUC     : {roc_auc_score(y_test, y_pred_prob):.4f}')
print(f'  Predicted churn: {y_pred.sum()} / {len(y_pred)}')
print(f'  Actual churn   : {y_test.sum()} / {len(y_test)}')

In [ ]:
print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred, target_names=['No Churn (0)', 'Churn (1)']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Task 4 — Confusion Matrix (Baseline)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correctly no churn): {tn}')
print(f'False Positives (wrong churn alarm)  : {fp}')
print(f'False Negatives (missed churners)    : {fn}')
print(f'True Positives  (caught churners)    : {tp}')

In [ ]:
print('--- Interpretation ---')
print("""
The dataset is heavily imbalanced (~98.45% no-churn, ~1.55% churn).
Raw accuracy is misleadingly high — even a model predicting 'no churn'
for everyone would score ~98%. Focus on:

  Recall (Churn)   : Did we catch the actual churners?
  Precision (Churn): When we predict churn, are we right?
  AUC-ROC          : Overall discrimination ability.

Class weights (63x for minority class) push the model to focus on churners.
""")

---
## Task 5: Hyperparameter Experimentation

In [ ]:
def run_experiment(name, hidden_layers=2, neurons=64, lr=0.001,
                   batch_size=32, epochs=80, activation='relu'):
    print(f"\n{'='*50}")
    print(f'  {name}')
    print(f"{'='*50}")
    m = build_model(hidden_layers=hidden_layers, neurons=neurons,
                    lr=lr, activation=activation)
    es = EarlyStopping(monitor='val_loss', patience=10,
                       restore_best_weights=True, verbose=0)
    hist = m.fit(X_train_scaled, y_train, epochs=epochs, batch_size=batch_size,
                 validation_split=0.15, class_weight=class_weight_dict,
                 callbacks=[es], verbose=0)
    epochs_run = len(hist.history['loss'])
    res = m.evaluate(X_test_scaled, y_test, verbose=0)
    y_prob = m.predict(X_test_scaled, verbose=0).flatten()
    auc_val = roc_auc_score(y_test, y_prob)
    y_p = (y_prob >= 0.5).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_test, y_p).ravel()
    prec = tp_ / (tp_ + fp_) if (tp_ + fp_) > 0 else 0
    rec  = tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0

    print(f'  Epochs run    : {epochs_run}')
    print(f'  Test Accuracy : {res[1]:.4f}')
    print(f'  AUC-ROC       : {auc_val:.4f}')
    print(f'  Recall(Churn) : {rec:.4f}  |  TP={tp_}  FN={fn_}')

    return {
        'Experiment': name,
        'Hidden Layers': hidden_layers,
        'Neurons': neurons,
        'Learning Rate': lr,
        'Batch Size': batch_size,
        'Activation': activation,
        'Epochs Run': epochs_run,
        'Test Accuracy': round(res[1], 4),
        'Test Loss': round(res[0], 4),
        'AUC-ROC': round(auc_val, 4),
        'Precision (Churn)': round(prec, 4),
        'Recall (Churn)': round(rec, 4),
        'True Positives': int(tp_),
        'False Negatives': int(fn_),
    }

In [ ]:
experiments = []

# Exp 1: Baseline
experiments.append(run_experiment('Exp 1 — Baseline',
    hidden_layers=2, neurons=64, lr=0.001, batch_size=32, activation='relu'))

# Exp 2: Deeper network
experiments.append(run_experiment('Exp 2 — Deep (4 layers)',
    hidden_layers=4, neurons=128, lr=0.001, batch_size=32, activation='relu'))

# Exp 3: High learning rate
experiments.append(run_experiment('Exp 3 — High LR (0.01)',
    hidden_layers=2, neurons=64, lr=0.01, batch_size=32, activation='relu'))

# Exp 4: Low learning rate
experiments.append(run_experiment('Exp 4 — Low LR (0.0001)',
    hidden_layers=2, neurons=64, lr=0.0001, batch_size=32, activation='relu'))

# Exp 5: Tanh activation + large batch
experiments.append(run_experiment('Exp 5 — Tanh + Large Batch',
    hidden_layers=2, neurons=64, lr=0.001, batch_size=128, activation='tanh'))

In [ ]:
results_df = pd.DataFrame(experiments)
display_cols = ['Experiment', 'Hidden Layers', 'Neurons', 'Learning Rate', 'Batch Size',
                'Activation', 'Epochs Run', 'Test Accuracy', 'AUC-ROC',
                'Precision (Churn)', 'Recall (Churn)', 'True Positives', 'False Negatives']
results_df = results_df[display_cols]
results_df.to_csv('results/model_comparison_table.csv', index=False)
print('Saved -> results/model_comparison_table.csv')
results_df

In [ ]:
# Styled comparison table image
fig, ax = plt.subplots(figsize=(18, 4))
ax.axis('off')
tbl = results_df.copy()
tbl['Learning Rate'] = tbl['Learning Rate'].astype(str)

t = ax.table(cellText=tbl.values, colLabels=tbl.columns, loc='center', cellLoc='center')
t.auto_set_font_size(False)
t.set_fontsize(8.5)
t.scale(1, 1.8)
for j in range(len(tbl.columns)):
    t[(0, j)].set_facecolor('#1565C0')
    t[(0, j)].set_text_props(color='white', fontweight='bold')
for i in range(1, len(tbl) + 1):
    clr = '#E3F2FD' if i % 2 == 0 else 'white'
    for j in range(len(tbl.columns)):
        t[(i, j)].set_facecolor(clr)

fig.suptitle('Task 5 — Hyperparameter Experiment Comparison', fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('results/model_comparison_table.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/model_comparison_table.png')

In [ ]:
# Bar chart comparison
exp_labels = [f'Exp {i+1}' for i in range(len(results_df))]
palette = ['#2196F3', '#4CAF50', '#F44336', '#FF9800', '#9C27B0']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Task 5 — Experiment Comparison', fontsize=13, fontweight='bold')

axes[0].bar(exp_labels, results_df['AUC-ROC'], color=palette, edgecolor='black')
axes[0].set_title('AUC-ROC by Experiment')
axes[0].set_ylim(0, 1.05)
for i, v in enumerate(results_df['AUC-ROC']):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

axes[1].bar(exp_labels, results_df['Recall (Churn)'], color=palette, edgecolor='black')
axes[1].set_title('Recall (Churn class) by Experiment')
axes[1].set_ylim(0, 1.05)
for i, v in enumerate(results_df['Recall (Churn)']):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('results/experiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Combined evaluation outputs summary
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Task 4 & 5 — Evaluation Outputs Summary', fontsize=14, fontweight='bold')

# Target distribution
axes[0, 0].bar(['No Churn (0)', 'Churn (1)'], churn_counts.values,
               color=['#2196F3', '#F44336'], edgecolor='black')
axes[0, 0].set_title('Target Distribution')
axes[0, 0].set_ylabel('Count')

# Confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1],
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
axes[0, 1].set_title('Confusion Matrix — Baseline')
axes[0, 1].set_ylabel('Actual')
axes[0, 1].set_xlabel('Predicted')

# AUC comparison
axes[1, 0].bar(exp_labels, results_df['AUC-ROC'], color=palette, edgecolor='black')
axes[1, 0].set_title('AUC-ROC by Experiment')
axes[1, 0].set_ylim(0, 1.05)
for i, v in enumerate(results_df['AUC-ROC']):
    axes[1, 0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

# Recall comparison
axes[1, 1].bar(exp_labels, results_df['Recall (Churn)'], color=palette, edgecolor='black')
axes[1, 1].set_title('Churn Recall by Experiment')
axes[1, 1].set_ylim(0, 1.05)
for i, v in enumerate(results_df['Recall (Churn)']):
    axes[1, 1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('results/evaluation_outputs.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/evaluation_outputs.png')

---
## Task 6: Final Reflection

### Q1. What role do weights and biases play in the model?

**Weights** are learnable parameters that control the strength of connection between neurons. Each weight multiplies an input feature — a large positive weight strongly activates a neuron for that feature, while a large negative weight suppresses it. During training, weights are updated via backpropagation to minimize the loss.

**Biases** are additional learned parameters that shift the activation threshold of each neuron. Without bias, the decision boundary would always pass through the origin, severely limiting the model's expressive power.

The key equation: `output = activation(W · X + b)` — weights scale inputs, bias shifts them.

---

### Q2. Why is an activation function required?

Without activation functions, stacking multiple linear layers is mathematically equivalent to a single linear transformation — no matter how many layers you add. Non-linear activations like **ReLU** allow the network to:

- Learn complex, curved decision boundaries
- Model non-linear relationships (e.g., churn depends on satisfaction AND tenure AND payment delay in non-trivial ways)
- Build hierarchical feature representations

In this model: **ReLU** in hidden layers (efficient, avoids vanishing gradient), **Sigmoid** in output layer (produces probability between 0 and 1).

---

### Q3. What happens when learning rate is too high or too low?

| Scenario | Effect |
|---|---|
| **Too High (0.01, Exp 3)** | Gradient steps overshoot the loss minimum. Loss may oscillate or diverge. Unstable training. |
| **Optimal (0.001, Exp 1)** | Steady convergence. Efficient learning without overshooting. |
| **Too Low (0.0001, Exp 4)** | Very slow convergence. More epochs needed. May stop too early with EarlyStopping before reaching minimum. |

The baseline lr=0.001 (Adam default) strikes the best balance for this dataset.

---

### Q4. Did the model show underfitting or overfitting?

Given the heavy class imbalance (~1.55% churn), the primary challenge is **class imbalance bias** rather than classic overfitting/underfitting:

- **Without class weights**: The model underfits the minority class — predicting "no churn" for almost everything, achieving high accuracy but near-zero recall on churners.
- **With class weights (~63:1)**: The model becomes more sensitive to churners, improving recall at the expense of some false positives.
- **Deeper models (Exp 2)** showed slight overfitting (val_loss diverges from train_loss), which EarlyStopping mitigated.
- The baseline with L2 + Dropout showed well-aligned train/val curves — **good generalization** for the dataset size.

**Conclusion:** The baseline model is well-calibrated. For production, the classification threshold (currently 0.5) could be lowered to prioritize catching more churners, depending on business cost of missed churn vs. false alarms.